<p style="padding: 10px; border: 1px solid black;">
<img src="images/mlu-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Getting Started with Multimodal Applications</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 4d: Multimodal Agents</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="images/mlu-activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="images/mlu-challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2E86C1; font-weight: bold; text-decoration: none;">1. Installing dependencies</a></p>
    <p><a href="#section2" style="color: #2E86C1; font-weight: bold; text-decoration: none;">2. Multimodal agent for image generation and description</a></p>
    <ul style="margin-top: 0; padding-left: 30px;">
        <li><a href="#section2-1" style="color: #3498DB; text-decoration: none;">2.1 Custom multimodal tools for image generation and description</a></li>
        <li><a href="#section2-2" style="color: #3498DB; text-decoration: none;">2.2 Agentic application for image generation and description</a></li>
    </ul>
    <p><a href="#section3" style="color: #2E86C1; font-weight: bold; text-decoration: none;">3. Multimodal agent for retrieval-based responses</a></p>
    <ul style="margin-top: 0; padding-left: 30px;">
        <li><a href="#section3-1" style="color: #3498DB; text-decoration: none;">3.1 Custom multimodal tools for retrieval</a></li>
        <li><a href="#section3-2" style="color: #3498DB; text-decoration: none;">3.2 Agentic application based on retrieval</a></li>
    </ul>
    <p><a href="#section4" style="color: #2E86C1; font-weight: bold; text-decoration: none;">4. Quizzes</a></p>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Installing dependencies</h2>
</div>

In this lab, we will develop custom multimodal tools and agents to accomplish certain complex tasks.

In [ ]:
%%capture
!pip install -q -r requirements.txt

Let's import the libraries and modules required for this lab. We will import the `invoke_nova_lite_multimodal` and `get_base64_encoded_image` functions we defined and used in previous labs.

In [ ]:
import sys
sys.path.append('..')

import boto3
import base64
import json
from IPython.display import JSON
import time
from tqdm import tqdm
from botocore.exceptions import ClientError
from IPython.display import Image, display, Markdown, IFrame
from langchain.tools import tool, BaseTool
import io
from PIL import Image as pil_image,  ImageDraw, ImageFont

from mlu_utils.multimodal_utils import invoke_nova_lite_multimodal, prepare_image, get_base64_encoded_image

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Multimodal agent for image generation and description</h2>
</div>

Let's see how we can develop a multimodal agent with custom tools for an engaging movie poster and story generation application.

1. **Movie poster generation**: You provide a prompt or concept for a movie, and the application utilizes the `image-generator-tool` to create an initial movie poster based on your input. This tool leverages the Stability AI Stable Diffusion 3.5 Large model to produce a visually compelling movie poster.

2. **Poster variation**: Once the first movie poster is generated, the `image_variation_tool` is employed to create a variation of the initial poster. This tool uses the Stability AI Control Structure service to produce a slightly different version of the movie poster, potentially representing a different genre, mood, or style.

3. **Story generation**: With the two movie posters in hand, the application then utilizes the `Image-to-story tool`, which is powered by the Amazon Nova Lite multimodal model. The multimodal agent analyzes the visual elements, symbolism, and imagery present in the movie posters and generates a compelling story or plot synopsis based on its understanding of the visual cues.

4. **Output**: The final output of the application is a set of two visually distinct movie posters and a corresponding story or plot synopsis that captures the essence and narrative suggested by the imagery. This combination of visual and language generation capabilities allows you to explore creative concepts and see how they might translate into compelling movie ideas.

The application leverages the strengths of different tools and models, including the Stability AI Stable Diffusion 3.5 Large and Control Structure services for visual generation and the Amazon Nova Lite model for visual understanding and language generation. By combining these capabilities, the application offers a unique and engaging experience for you to explore movie ideas and see how visual elements can inspire and shape narratives.

<!-- Subsection Header -->
<div id="section2-1" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">2.1 Custom multimodal tools for image generation and description</h3>
</div>

In [ ]:
@tool
def image_to_story_tool(image_path: str):
    """Use this tool to generate a story related to a given image. The input of the tool is the path of the image."""
    #image_string, image_type = get_base64_encoded_image(image_path.replace("\n", ""))
    image_path_clean = image_path.replace("\n", "").strip().strip('"').strip("'")
    # Remove keyword argument syntax like image_path="..."
    if '=' in image_path_clean:
        image_path_clean = image_path_clean.split('=', 1)[1].strip().strip('"').strip("'")
    # Remove any trailing ReAct artifacts the agent may append
    for suffix in ['Observation', 'Thought', 'Action', 'Final Answer']:
        if suffix in image_path_clean:
            image_path_clean = image_path_clean[:image_path_clean.index(suffix)].strip().strip('"').strip("'")
    image_binary, image_type = prepare_image(image_path_clean)
    prompt = "Write an interesting story related to the given image. Produce the response without a preamble. Just write the story."
    response = invoke_nova_lite_multimodal(prompt=prompt, images=image_binary, image_types=image_type) 
    return response

In [ ]:
@tool
def add_text_to_image(image_path):
    """Use this tool to add the title to the movie poster. The input is a string with the image_path and title separated by comma."""
    # Open the image
    image = pil_image.open(image_path)

    # Create a drawing object
    draw = ImageDraw.Draw(image)

    # Define the font and its properties
    font_path = "data/lab4/Agents/FranklinGothic.ttf"  # Replace this with the path to your desired font file
    font_size = 80  # Adjust the font size as needed
    font = ImageFont.truetype(font_path, font_size)

    # Calculate the text position
    text_width = draw.textlength(text, font)
    image_width, image_height = image.size
    text_x = (image_width - text_width) / 2  # Center the text horizontally
    text_y = image_height - font_size - 50  # Position the text near the bottom

    # Draw the text on the image
    draw.text((text_x, text_y), text, font=font, fill=(255, 255, 255))  # White text color

    # Save the modified image
    image.save(output_path)

In [ ]:
@tool
def image_generator_tool(prompt: str) -> str:
    """
    Generate an image using a text prompt.
    
    Args:
        prompt (str): The text prompt for image generation.
        
    Returns:
        str: The file path of the generated image.
    """
    
    # Initialize AWS client for Bedrock Runtime
    client = boto3.client(service_name="bedrock-runtime", region_name="us-west-2")
    
    # Set request headers
    accept = "application/json"
    content_type = "application/json"
    
    # Set model ID for image generation
    model_id = 'stability.sd3-5-large-v1:0'
    
    # Prepare request body
    body = json.dumps({
        "prompt": prompt
    })
    
    # Invoke the model
    response = client.invoke_model(
        body=body, modelId=model_id, accept=accept, contentType=content_type
    )
    
    # Parse the response
    response_body = json.loads(response.get("body").read())
    finish_reason = response_body.get('finish_reasons', [None])[0]
    if finish_reason is not None:
        raise Exception(f"Image generation error: {finish_reason}")
    img = response_body.get('images')[0]
    base64_bytes = img.encode('ascii')
    image_bytes = base64.b64decode(base64_bytes)
    
    # Save the generated image
    image_path = "generated_image.png"
    pil_image.open(io.BytesIO(image_bytes)).save(image_path)
    
    return image_path

In [ ]:
@tool
def image_variation_tool(prompt: str):
    """
    Generate a second, alternative version of a movie poster. Use this tool AFTER image_generator_tool to create a different poster for the same movie. The input is the text prompt describing the movie poster.

    Args:
        prompt (str): The text prompt for the alternative poster.

    Returns:
        str: The file path of the generated variation image.
    """

    # Initialize the AWS Bedrock Runtime client
    client = boto3.client(service_name="bedrock-runtime", region_name="us-west-2")

    # Set the request headers and parameters
    accept = "application/json"
    content_type = "application/json"
    image_path = 'generated_image.png'
    model_id = 'stability.sd3-5-large-v1:0'

    # Create the request body - generate a variation using a modified prompt
    variation_prompt = f"A different artistic interpretation of: {prompt}. Alternative style, different color palette, unique composition."
    body = json.dumps({
        "prompt": variation_prompt
    })

    # Invoke the Bedrock Runtime model for image variation
    response = client.invoke_model(
        body=body, modelId=model_id, accept=accept, contentType=content_type
    )
    response_body = json.loads(response.get("body").read())

    # Extract the generated image from the response
    finish_reason = response_body.get('finish_reasons', [None])[0]
    if finish_reason is not None:
        raise Exception(f"Image variation error: {finish_reason}")
    img = response_body.get('images')[0]
    base64_bytes = img.encode('ascii')
    image_bytes = base64.b64decode(base64_bytes)

    # Save the generated image to a file
    output_path = "variation_image.png"
    pil_image.open(io.BytesIO(image_bytes)).save(output_path)

    return output_path

<!-- Subsection Header -->
<div id="section2-2" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">2.2 Agentic application for image generation and description</h3>
</div>

Let's define the custom agent that will select the best tool at each planning step using the ReAct logic and accomplish the task. The application utilizes LangChain's Agent Executor to orchestrate the custom agentic workflow that leverages three tools: an image generator, an image variation tool, and an image-to-story tool.

The agent workflow proceeds as follows:

1. Ingest the user's movie concept or prompt.
2. Invoke the image generator tool to create an initial movie poster.
3. Call the image variation tool to generate a second poster variation.
4. Utilize the image-to-story tool (powered by a multimodal model) to analyze the visual elements of both posters and generate a corresponding plot synopsis.
5. Collate and present the two movie posters and the generated plot synopsis as the final output.

The agent executor acts as the central coordinator, managing the execution flow and data transfer between the custom tools. This agentic approach, facilitated by LangChain, enables the seamless integration and orchestration of the custom tools, resulting in a streamlined process for generating movie posters, variations, and narratives based on the user's input.

In [ ]:
# define custom agent
def create_custom_agent(tools):
    """
    Creates a custom agent with the given tools and a specific prompt template.

    Args:
        tools (list): A list of tools to be used by the agent.

    Returns:
        AgentExecutor: An instance of the AgentExecutor class with the custom agent.
    """
    import re
    from langchain_aws import ChatBedrockConverse
    from langchain.agents import AgentExecutor, create_react_agent
    from langchain_core.prompts.chat import ChatPromptTemplate
    from langchain.agents.output_parsers import ReActSingleInputOutputParser
    from langchain_core.agents import AgentAction, AgentFinish
    
    class FixedReActOutputParser(ReActSingleInputOutputParser):
        """Custom parser that handles function-call-style actions like tool_name(args)."""
        def parse(self, text: str):
            # Fix function-call-style actions: tool_name(args) -> tool_name + args
            func_call_pattern = r'Action:\s*(\w+)\((.*)\)'
            match = re.search(func_call_pattern, text, re.DOTALL)
            if match:
                tool_name = match.group(1).strip()
                tool_input = match.group(2).strip().strip('"').strip("'")
                # Remove keyword argument syntax like key="value"
                if '=' in tool_input and not any(c in tool_input.split('=')[0] for c in ' /\\.'):
                    tool_input = tool_input.split('=', 1)[1].strip().strip('"').strip("'")
                text = text[:match.start()] + f'Action: {tool_name}\nAction Input: {tool_input}'
            return super().parse(text)
    
    # Initialize the large language model (LLM) with the specified model ID and temperature
    llm = ChatBedrockConverse(
        model="amazon.nova-pro-v1:0",
        temperature=0
    )

    # Define the prompt template for the agent
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Answer the following questions as best you can. You have access to the following tools:

                {tools}

                Use the following format:

                Question: the input question you must answer
                Thought: you should always think about what to do
                Action: the action to take, should be one of [{tool_names}]
                Action Input: the input to the action
                Observation: the result of the action
                ... (this Thought/Action/Action Input/Observation can repeat N times)
                Thought: I now know the final answer
                Final Answer: the final answer to the original input question""",
            ),
            ("user", "Begin!\n\nQuestion: {input}\nThought:{agent_scratchpad}")
        ]
    )

    #########################

    # Create the custom agent using the LLM, tools, and prompt
    agent = create_react_agent(llm, tools, prompt, output_parser=FixedReActOutputParser())

    # Create an instance of the AgentExecutor with the custom agent
    return AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True, handle_parsing_errors=True, max_iterations=10)

In [ ]:
# Create a list of all relevant tools
tools = [image_to_story_tool, image_generator_tool, image_variation_tool]

# Define the custom agent and provide the agent access to the above tools
movie_agent = create_custom_agent(tools)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Try it yourself!</h4>
        <p>Try different prompts and observe the posters and the plots of the movies using the custom agent.</p>
    </div>
</div>

In [ ]:
# Test out the agent with a simple example
prompt = """Draw a poster for a sci-fi PG-13 movie called 'Paradox' using the image_generator_tool, \
then create a second different poster for the same movie using the image_variation_tool, \
and finally write the story of the movie based on the first poster using the image_to_story_tool."""

response_movie = movie_agent.invoke({"input": prompt})

#### Here's the first movie poster:

In [ ]:
Image("generated_image.png", width=300)

#### Here's the second movie poster:

In [ ]:
Image("variation_image.png", width=300)

#### And here's the plot of the movie:

In [ ]:
Markdown("<i>"+ response_movie['intermediate_steps'][-1][1] +"</i>")

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Use all the tools</h4>
        <p>In the example above, we did not need to use the <code>add_text_to_image</code> tool. Try to craft a prompt so that this tool is also used.</p>
    </div>
</div>

In [ ]:
### Enter your code below




###

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Multimodal agent for retrieval-based responses</h2>
</div>

Here is a description of the multimodal agent for retrieval-based workflows:

This multimodal AI agent is designed to assist in verifying the authenticity of physical products by leveraging web search capabilities and advanced multimodal techniques. The agent has access to three custom tools, discussed in the next section.

<!-- Subsection Header -->
<div id="section3-1" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">3.1 Custom multimodal tools for retrieval</h3>
</div>

Let's define three custom tools to allow the agent to retrieve results from websites to authenticate the product in the image as well as suggest websites where the authentic image may be purchased.

1. **Image Comparison Tool**: This tool utilizes Amazon Nova's state-of-the-art multimodal capabilities to compare two product images in detail. It can analyze and compare various visual properties, such as shape, color, design elements, and intricate details, to determine the similarity or dissimilarity between the images.

2. **Product Web Search**: This tool allows the agent to perform comprehensive web searches using DuckDuckGo to gather information and visual representations of a specific product. It can retrieve product descriptions, specifications, and images from various online sources, building a comprehensive knowledge base about the product.

3. **Image Web Search**: This tool enables the agent to search the web for images of a product based on a text prompt. It can find and retrieve relevant product images from various online sources, further enhancing the agent's visual knowledge base.

In [ ]:
@tool
def image_comparison(image_paths:str):
    """Use this tool to compare and contrast two images. The input of the tool is a string consisting of both image paths seperated by comma. Image paths can be local paths or urls."""
    image_paths_arr = [f.strip().replace("\n", "").strip('"').strip("'") for f in image_paths.split(',')]
    image_binary, image_type = prepare_image(image_paths_arr)
    prompt = "Compare and contrast the two images. Share insights on if they are completely identical, similar or distinct. Produce the response without a preamble. Just write the analysis."
    response = invoke_nova_lite_multimodal(prompt=prompt, images=image_binary, image_types=image_type)

    return response

In [ ]:
from ddgs import DDGS

@tool
def product_web_search(prompt:str):
    """Search online for a website about a product. The input is the prompt with the product ID and the brand name. The prompt needs to be under 45 characters."""
    search_tool = DDGS()
    time.sleep(1)  # Add delay between requests
    response = search_tool.text(query=prompt, max_results=5, region='us-en', safesearch='on')
    
    return response


@tool
def image_web_search(prompt:str):
    """Search the web for images of a product based on the prompt. The input is the prompt or query used to search for images of a product."""
    search_tool = DDGS()
    time.sleep(10)  # Add delay between requests
    response = search_tool.images(query=prompt, region='us-en', max_results=1)[0]['image'].partition("?")[0]
    time.sleep(10)  # Add delay between requests
    return response


<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Create custom tools</h4>
        <p>The agent's ability to pick the appropriate tool is crucial in achieving the desired response. Let's explore this ability by providing the agent with many more tools. Create a few more useful tools in the cell below and test the agent's response when it has to select from many more tools.</p>
    </div>
</div>

In [ ]:
### Enter your code below




###

<!-- Subsection Header -->
<div id="section3-2" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">3.2 Agentic application based on retrieval</h3>
</div>

Let's define the custom agent, similar to the previous example, that will select the best tool at each planning step using the ReAct logic. 

The multimodal agent's workflow is as follows:

1. When presented with an image of a product, the agent utilizes the image web search tool to find additional images of the product, expanding its visual knowledge base.

2. Using the image comparison tool, the agent compares the visual information gathered from the web with the physical product in question. 

3. Based on the comparison results, the agent can provide an assessment of whether the physical product is likely to be authentic or an imitation.

4. Finally it will search online for a website where the user may purchase the authenticated product.

This multimodal agent leverages the power of web search, computer vision, and multimodal analysis to provide a comprehensive solution for product authentication. By combining textual and visual information from various online sources with advanced image comparison techniques, the agent can assist in verifying the authenticity of physical products with a high degree of accuracy and reliability.

In [ ]:
retrieval_tools = [product_web_search, image_web_search, image_comparison]
retrieval_agent = create_custom_agent(retrieval_tools)

In [ ]:
Image("content/Agents/nike.jpg", width=300)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Try it yourself!</h4>
        <p>Try different images of products and observe how the agent authenticates the product using the tools.</p>
    </div>
</div>

If you get a `RateLimitException`, wait a few minutes before trying again. DuckDuckGO is a free web search API and limits the number of API requests.

In [ ]:
prompt = """I have an image of a shoe at "./content/Agents/nike.jpg".  
Can you check if they are Nike Air Jordan 1 Low SE FN5214-131? 
If they are, find the link to the website where i can find the product. Do not generate clickable links in the output."""
try:
    response_shoes = retrieval_agent.invoke({"input":prompt})
except Exception as e:
    response_shoes = {}
    if "403" in str(e):
        print(f"\nRatelimitException raised. Wait some minutes before trying again")
    else:
        print(f"\nAn unexpected error occurred: {str(e)}")

#### Here's the response about the authenticity of the product:

### ⚠️ IMPORTANT SECURITY NOTICE ⚠️
The URLs displayed in this educational notebook are REAL URLs. While they are shown for educational purposes, we strongly advise:

- DO NOT click on or visit these URLs
- DO NOT use them for further exploration
- DO NOT assume they are safe or vetted

This notebook is for demonstration purposes only. Visiting unknown URLs can expose you to security risks, malware, or inappropriate content. Always practice safe browsing habits and only visit trusted, verified websites.

If you need to explore web resources, please use official documentation and trusted sources.

In [ ]:
if 'output' in response_shoes:
    result = Markdown("<i>"+response_shoes['output'] + "</i>")
else:
    result = Markdown("No field `output` found in response data")
result

#### Here's the response about the webpage to purchase the original product.

In [ ]:
if 'intermediate_steps' in response_shoes and response_shoes['intermediate_steps']:
    url = response_shoes['intermediate_steps'][-1][1]
    if isinstance(url, str):
        Markdown(f"<i>{url}</i>")
    else:
        JSON(url)
else:
    Markdown("No intermediate steps found in response data")

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">4. Quizzes</h2>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Try it Yourself!</h4>
        <p>Answer the following questions to test your understanding of using multimodal models for generating personalized and inclusive content.</p>
    </div>
</div>

In [ ]:
from mlu_utils.quiz_questions import lab5c_question1, lab5c_question2

lab5c_question1.display()
lab5c_question2.display()

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have:</p>
    <ul>
        <li>Developed custom multimodal tools for image generation and description</li>
        <li>Created an agentic application for movie poster generation and storytelling</li>
        <li>Built custom multimodal tools for retrieval-based responses</li>
        <li>Implemented a product authentication agent using web search and image comparison</li>
    </ul>
    <h4 style="color: #2874A6; margin-top: 15px;">Additional Resources</h4>
    <ul>
        <li><a href="https://python.langchain.com/docs/modules/agents/">LangChain Agents Documentation</a></li>
        <li><a href="https://aws.amazon.com/bedrock/">Amazon Bedrock Documentation</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="images/mlu-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!